主要实现内容：合并基于近15天销量、近30天销量、运营提供的参数生成的发货建议

In [1]:
import pandas as pd
import openpyxl
import warnings

In [2]:
parms = '20260518'
send_goods15 = pd.read_excel(f'../程序建议发货列表/发货列表{parms}-new-7.0-120-{parms}-15.xlsx')
send_goods30 = pd.read_excel(f'../程序建议发货列表/发货列表{parms}-new-7.0-120-{parms}-30.xlsx')
send_goods60 = pd.read_excel(f'../程序建议发货列表/发货列表{parms}-new-7.0-120-{parms}-60.xlsx')
send_goods = pd.read_excel(f'../程序建议发货列表/发货列表{parms}-new-7.0-120-模板.xlsx')

# shipment_path = f'../src_data/在途货件/在途货件{parms}.xlsx'

In [3]:
shipment_path = f'../src_data/在途货件/在途货件{parms}.xlsx'

In [4]:
shipment_receive_total = pd.read_excel(shipment_path, sheet_name='接收中-汇总')
shipment_receive_detail = pd.read_excel(shipment_path, sheet_name='接收中-货件详情')

In [5]:
send_goods.columns

Index(['店铺-站点', '站点', '负责人', 'Listing', '款式', 'MSKU', '积加SKU', 'FNSKU', '仓储类型',
       '规模定位', '需求定位', '发货定位', '备货定位', '总发货天数', '快递在途天数', '空运在途天数', '海运在途天数',
       '快递打包时间', '空运打包时间', '海运打包时间', '补货', '借调', 'FBA在库可售天数', 'FBA总可售天数',
       '本地库存可售天数', '总库存可售天数', '预估在库日均', '断货风险总天数', '断货总损失销量', '首次断货前可售天数',
       'FBA库存', '已出运', 'FBA预占', 'FBA在途', 'dhl_pre', 'air_pre', 'sea_pre',
       'dhl_pre实际可发货数', 'air_pre实际可发货数', 'sea_pre实际可发货数', 'dhl_pre缺货数',
       'air_pre缺货数', 'sea_pre缺货数', '预占总数', '快递_预占', '空运_预占', '海运_预占',
       'FBA自提物流_预占', 'CA', 'DE', 'JP', 'UK', 'US', 'TK本地仓', '共享', '本地-在途',
       '已下单数量', '已生产未发货', 'CA仓搜海外仓', 'DE商易海外仓', 'DE延讯海外仓', 'JP永翔海外仓',
       'UK商易海外仓', 'UK延讯海外仓', 'US商易海外仓', 'US易速达海外仓', '九方欧洲海外仓', '元坤海外仓',
       'IT永翔海外仓', 'CN易速达:易速达美东GA仓', '顺丰SF:美国特拉华S2仓', '顺丰SF:美国洛杉矶S5仓',
       '顺丰SF:美国达拉斯C1仓', '顺丰SF:美国芝加哥C1仓', '7天日均', '15天日均', '30天日均', '断货时间1',
       '断货总天数1', '损失销量1', '断货开始天数1', '断货时间2', '断货总天数2', '损失销量2', '断货开始天数2',
       '断货时间3', '断货总天数3', '损失销量3', '断货

In [6]:
select_columns = ['店铺-站点', '站点', '负责人', 'Listing', '款式', 'MSKU', '积加SKU', 'FNSKU', '仓储类型', '规模定位', '需求定位', '发货定位', '备货定位', '补货', '借调', 
                  'FBA在库可售天数', 'FBA总可售天数', '预估在库日均', '首次断货前可售天数', 
                  '总发货天数', '快递在途天数', '空运在途天数', '海运在途天数', '快递打包时间', '空运打包时间', '海运打包时间',
                  # 'FBA库存', '已出运','FBA预占','FBA在途', '预占总数', '快递_预占', '空运_预占', '海运_预占', 
                  'FBA库存', '已出运','FBA预占','FBA在途', '预占总数', 'FBA自提物流_预占', '快递_预占', '空运_预占', '海运_预占', 
                  'dhl_pre', 'air_pre', 'sea_pre', 'dhl_pre实际可发货数', 'air_pre实际可发货数', 'sea_pre实际可发货数', 'dhl_pre缺货数', 'air_pre缺货数', 'sea_pre缺货数', 
                  'CA', 'DE', 'JP', 'UK', 'US','TK本地仓', '共享', '本地-在途', '已下单数量', '已生产未发货',
                  'CA仓搜海外仓','DE商易海外仓', 'DE延讯海外仓', 'JP永翔海外仓', 'UK商易海外仓', 'UK延讯海外仓', 'US商易海外仓', 'US易速达海外仓', '九方欧洲海外仓', '元坤海外仓', 'IT永翔海外仓', 'CN易速达:易速达美东GA仓',
                  '顺丰SF:美国特拉华S2仓', '顺丰SF:美国洛杉矶S5仓', '顺丰SF:美国达拉斯C1仓', '顺丰SF:美国芝加哥C1仓',
                  '7天日均', '15天日均', '30天日均',
                  '断货时间1', '断货总天数1', '损失销量1', '断货开始天数1', '断货时间2', '断货总天数2', '损失销量2', '断货开始天数2', '断货时间3', '断货总天数3', '损失销量3', '断货开始天数3']
# '断货时间4', '断货总天数4','损失销量4', '断货开始天数4'
# '断货时间5', '断货总天数5','损失销量5', '断货开始天数5'
send_goods = send_goods[select_columns]

In [7]:
select_columns2 = ['补货', '借调', 
                  'FBA在库可售天数', 'FBA总可售天数', '预估在库日均', '首次断货前可售天数', 
                  'dhl_pre', 'air_pre', 'sea_pre', 'dhl_pre实际可发货数', 
                  'air_pre实际可发货数', 'sea_pre实际可发货数',                                                                       
                  'dhl_pre缺货数', 'air_pre缺货数', 'sea_pre缺货数']

In [8]:
send_goods15 = send_goods15[select_columns2]
rename_dict = {value: f'{value}_15' for value in select_columns2}
send_goods15.rename(columns=rename_dict, inplace=True)
send_goods15

,补货_15,借调_15,FBA在库可售天数_15,FBA总可售天数_15,预估在库日均_15,首次断货前可售天数_15,dhl_pre_15,air_pre_15,sea_pre_15,dhl_pre实际可发货数_15,air_pre实际可发货数_15,sea_pre实际可发货数_15,dhl_pre缺货数_15,air_pre缺货数_15,sea_pre缺货数_15
0,该SKU销占比为0,否,0,0,0.00,0,0,0,0,0,0,0,0,0,0
1,是,是,32,55,0.53,55,0,5,16,0,0,0,0,5,16
2,否,否,165,135,0.48,82,0,0,0,0,0,0,0,0,0
3,否,否,76,82,1.03,82,0,0,0,0,0,0,0,0,0
4,是,否,43,47,0.56,47,0,11,16,0,11,16,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3690,该SKU销占比为0,否,0,0,0.00,0,0,0,0,0,0,0,0,0,0
3691,是,否,7,7,0.14,7,4,12,8,0,0,0,4,12,8
3692,是,否,17,17,1.06,17,18,76,48,0,0,0,18,76,48
3693,是,否,56,56,0.30,56,0,1,8,0,0,0,0,1,8


In [9]:
send_goods30 = send_goods30[select_columns2]
rename_dict = {value: f'{value}_30' for value in select_columns2}
send_goods30.rename(columns=rename_dict, inplace=True)
send_goods30

,补货_30,借调_30,FBA在库可售天数_30,FBA总可售天数_30,预估在库日均_30,首次断货前可售天数_30,dhl_pre_30,air_pre_30,sea_pre_30,dhl_pre实际可发货数_30,air_pre实际可发货数_30,sea_pre实际可发货数_30,dhl_pre缺货数_30,air_pre缺货数_30,sea_pre缺货数_30
0,该SKU销占比为0,否,0,0,0.00,0,0,0,0,0,0,0,0,0,0
1,否,否,48,83,0.35,82,0,0,0,0,0,0,0,0,0
2,否,否,165,135,0.48,82,0,0,0,0,0,0,0,0,0
3,是,是,64,67,1.22,67,0,0,20,0,0,0,0,0,20
4,是,否,63,69,0.38,69,0,0,5,0,0,5,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3690,是,否,0,0,0.00,0,1,4,2,0,0,0,1,4,2
3691,是,否,6,6,0.17,6,5,15,9,0,0,0,5,15,9
3692,是,否,20,20,0.90,20,9,56,35,0,0,0,9,56,35
3693,是,否,52,52,0.33,52,0,3,8,0,0,0,0,3,8


In [10]:
send_goods60 = send_goods60[select_columns2]
rename_dict = {value: f'{value}_60' for value in select_columns2}
send_goods60.rename(columns=rename_dict, inplace=True)
send_goods60

,补货_60,借调_60,FBA在库可售天数_60,FBA总可售天数_60,预估在库日均_60,首次断货前可售天数_60,dhl_pre_60,air_pre_60,sea_pre_60,dhl_pre实际可发货数_60,air_pre实际可发货数_60,sea_pre实际可发货数_60,dhl_pre缺货数_60,air_pre缺货数_60,sea_pre缺货数_60
0,该SKU销占比为0,否,0,0,0.00,0,0,0,0,0,0,0,0,0,0
1,否,否,49,86,0.35,82,0,0,0,0,0,0,0,0,0
2,否,否,102,109,0.77,82,0,0,0,0,0,0,0,0,0
3,否,否,77,83,1.01,82,0,0,0,0,0,0,0,0,0
4,是,否,56,61,0.43,61,0,0,11,0,0,11,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3690,是,否,0,0,0.00,0,12,33,20,0,0,0,12,33,20
3691,是,否,0,0,inf,0,58,149,94,0,0,0,58,149,94
3692,是,否,19,19,0.95,19,11,59,37,0,0,0,11,59,37
3693,是,否,40,40,0.42,40,0,12,12,0,0,0,0,12,12


In [11]:
ret_df60 = pd.concat(objs=[send_goods, send_goods60], axis=1)
ret_df60['在库天数差'] = ret_df60['FBA在库可售天数'] - ret_df60['FBA在库可售天数_60']
ret_df60['在库日均比'] = (ret_df60['预估在库日均'] / ret_df60['预估在库日均_60']).fillna(0).round(2)
ret_df60 = pd.merge(left=ret_df60, right=shipment_receive_total[['店铺-站点', 'MSKU', '差异量']], how='left', on=['店铺-站点', 'MSKU'])
ret_df60['差异量'] = ret_df60['差异量'].fillna(0).astype(int)

ret_df30 = pd.concat(objs=[send_goods, send_goods30], axis=1)
ret_df30['在库天数差'] = ret_df30['FBA在库可售天数'] - ret_df30['FBA在库可售天数_30']
ret_df30['在库日均比'] = (ret_df30['预估在库日均'] / ret_df30['预估在库日均_30']).fillna(0).round(2)
ret_df30 = pd.merge(left=ret_df30, right=shipment_receive_total[['店铺-站点', 'MSKU', '差异量']], how='left', on=['店铺-站点', 'MSKU'])
ret_df30['差异量'] = ret_df30['差异量'].fillna(0).astype(int)

ret_df15 = pd.concat(objs=[send_goods, send_goods15], axis=1)
ret_df15['在库天数差'] = ret_df15['FBA在库可售天数'] - ret_df15['FBA在库可售天数_15']
ret_df15['在库日均比'] = (ret_df15['预估在库日均'] / ret_df15['预估在库日均_15']).fillna(0).round(2)
ret_df15 = pd.merge(left=ret_df15, right=shipment_receive_total[['店铺-站点', 'MSKU', '差异量']], how='left', on=['店铺-站点', 'MSKU'])
ret_df15['差异量'] = ret_df15['差异量'].fillna(0).astype(int)

In [12]:
ret_df30.columns

Index(['店铺-站点', '站点', '负责人', 'Listing', '款式', 'MSKU', '积加SKU', 'FNSKU', '仓储类型',
       '规模定位',
       ...
       'sea_pre_30', 'dhl_pre实际可发货数_30', 'air_pre实际可发货数_30',
       'sea_pre实际可发货数_30', 'dhl_pre缺货数_30', 'air_pre缺货数_30', 'sea_pre缺货数_30',
       '在库天数差', '在库日均比', '差异量'],
      dtype='object', length=103)

In [13]:
ret_columns15 = ['店铺-站点', '站点', '负责人', 'Listing', '款式', 'MSKU', '积加SKU', 'FNSKU', '仓储类型',
       '规模定位', '需求定位', '发货定位', '备货定位','总发货天数', '快递在途天数', '空运在途天数', '海运在途天数', '快递打包时间', '空运打包时间', '海运打包时间',
       '补货', '借调', '补货_15','借调_15', 
       'FBA在库可售天数', 'FBA总可售天数', 'FBA在库可售天数_15', 'FBA总可售天数_15', 
       '预估在库日均', '预估在库日均_15', '在库天数差', '在库日均比','首次断货前可售天数', '首次断货前可售天数_15',
       'FBA库存', '已出运', 'FBA预占','FBA在途', '差异量',
       'dhl_pre', 'air_pre', 'sea_pre', 'dhl_pre_15', 'air_pre_15', 'sea_pre_15', 
       'dhl_pre实际可发货数', 'air_pre实际可发货数','sea_pre实际可发货数', 'dhl_pre实际可发货数_15', 'air_pre实际可发货数_15', 'sea_pre实际可发货数_15', 
       'dhl_pre缺货数', 'air_pre缺货数', 'sea_pre缺货数', 'dhl_pre缺货数_15','air_pre缺货数_15', 'sea_pre缺货数_15',
       # '预占总数', '快递_预占', '空运_预占', '海运_预占', 
       '预占总数', 'FBA自提物流_预占', '快递_预占', '空运_预占', '海运_预占', 
       'CA', 'DE','JP', 'UK', 'US', 
       'TK本地仓', '共享', '本地-在途', '已下单数量','已生产未发货',
       'CA仓搜海外仓', 'DE商易海外仓', 'DE延讯海外仓','JP永翔海外仓', 'UK商易海外仓', 'UK延讯海外仓', 'US商易海外仓', 'US易速达海外仓', '九方欧洲海外仓','元坤海外仓', 'IT永翔海外仓', 'CN易速达:易速达美东GA仓',
       '顺丰SF:美国特拉华S2仓', '顺丰SF:美国洛杉矶S5仓', '顺丰SF:美国达拉斯C1仓', '顺丰SF:美国芝加哥C1仓',
       '7天日均', '15天日均', '30天日均', 
       '断货时间1', '断货总天数1', '损失销量1','断货开始天数1', '断货时间2', '断货总天数2', '损失销量2', '断货开始天数2', '断货时间3', '断货总天数3','损失销量3', '断货开始天数3'
       ]

# # 指定要添加颜色的列
# ret_columns15_color = {'店铺-站点': 'FFD7E3BD', '站点': 'FFD7E3BD', '负责人': 'FFD7E3BD', 'Listing': 'FFD7E3BD', 
#                 '款式': 'FFD7E3BD', 'MSKU': 'FFD7E3BD', '积加SKU': 'FFD7E3BD', 'FNSKU': 'FFD7E3BD', '仓储类型': 'FFD7E3BD','规模定位': 'FFD7E3BD', 
#                 '需求定位': 'FFD7E3BD', '发货定位': 'FFD7E3BD', '备货定位': 'FFD7E3BD', 
#                 '补货': 'FFFDE9D9', '借调': 'FFFDE9D9', '补货_15': 'FFFCD5B4', '借调_15': 'FFFCD5B4', 
#                 'FBA在库可售天数': 'FFDAEEF3', 'FBA总可售天数': 'FFDAEEF3', 'FBA在库可售天数_15': 'FFB7DEE8', 'FBA总可售天数_15': 'FFB7DEE8', 
#                 '预估在库日均': 'FFE4DFEC', '预估在库日均_15': 'FFCCC0DA', '在库天数差': 'FFD7E3BD', '在库日均比': 'FFA9D18E','首次断货前可售天数': 'FFF2DCDB', '首次断货前可售天数_15': 'FFE6B8B7',
#                 'FBA库存': 'FFB8CCE4', '已出运': 'FFB8CCE4', 'FBA预占': 'FFB8CCE4','FBA在途': 'FFB8CCE4', '差异量': 'FFE6B8B7',
#                 'dhl_pre': 'FFFDE9D9', 'air_pre': 'FFFDE9D9', 'sea_pre': 'FFFDE9D9', 'dhl_pre_15': 'FFFCD5B4', 'air_pre_15': 'FFFCD5B4', 'sea_pre_15': 'FFFCD5B4', 
#                 'dhl_pre实际可发货数': 'FFDAEEF3', 'air_pre实际可发货数': 'FFDAEEF3','sea_pre实际可发货数': 'FFDAEEF3', 'dhl_pre实际可发货数_15': 'FFB7DEE8', 'air_pre实际可发货数_15': 'FFB7DEE8', 'sea_pre实际可发货数_15': 'FFB7DEE8', 
#                 'dhl_pre缺货数': 'FFE4DFEC', 'air_pre缺货数': 'FFE4DFEC', 'sea_pre缺货数': 'FFE4DFEC', 'dhl_pre缺货数_15': 'FFE4DFEC','air_pre缺货数_15': 'FFE4DFEC', 'sea_pre缺货数_15': 'FFE4DFEC',
#                 'CA': 'FFD7E3BD', 'DE': 'FFD7E3BD','JP': 'FFD7E3BD', 'UK': 'FFD7E3BD', 'US': 'FFD7E3BD', '共享': 'FFD7E3BD', '本地-在途': 'FFF2DCDB', '已下单数量': 'FFE6B8B7',
#                 'CA仓搜海外仓': 'FFB8CCE4', 'DE商易海外仓': 'FFB8CCE4', 'DE延讯海外仓': 'FFB8CCE4','JP永翔海外仓': 'FFB8CCE4', 'UK商易海外仓': 'FFB8CCE4', 'UK延讯海外仓': 'FFB8CCE4', 'US商易海外仓': 'FFB8CCE4', 'US易速达海外仓': 'FFB8CCE4', '九方欧洲海外仓': 'FFB8CCE4','元坤海外仓': 'FFB8CCE4', 
#                 '7天日均': 'FFFCD5B4', '15天日均': 'FFFCD5B4', '30天日均': 'FFFCD5B4', 
#                 '断货时间1': 'FFD7E3BD', '断货总天数1': 'FFD7E3BD', '损失销量1': 'FFD7E3BD','断货开始天数1': 'FFD7E3BD', '断货时间2': 'FFD7E3BD', '断货总天数2': 'FFD7E3BD', '损失销量2': 'FFD7E3BD', '断货开始天数2': 'FFD7E3BD', '断货时间3': 'FFD7E3BD', '断货总天数3': 'FFD7E3BD','损失销量3': 'FFD7E3BD', '断货开始天数3': 'FFD7E3BD'
#                       }

In [14]:
ret_columns30 = ['店铺-站点', '站点', '负责人', 'Listing', '款式', 'MSKU', '积加SKU', 'FNSKU', '仓储类型',
       '规模定位', '需求定位', '发货定位', '备货定位', '总发货天数', '快递在途天数', '空运在途天数', '海运在途天数', '快递打包时间', '空运打包时间', '海运打包时间',
       '补货', '借调', '补货_30','借调_30', 
       'FBA在库可售天数', 'FBA总可售天数', 'FBA在库可售天数_30', 'FBA总可售天数_30', 
       '预估在库日均', '预估在库日均_30', '在库天数差', '在库日均比','首次断货前可售天数', '首次断货前可售天数_30',
       'FBA库存', '已出运', 'FBA预占','FBA在途','差异量',
       'dhl_pre', 'air_pre', 'sea_pre', 'dhl_pre_30', 'air_pre_30', 'sea_pre_30', 
       'dhl_pre实际可发货数', 'air_pre实际可发货数','sea_pre实际可发货数', 'dhl_pre实际可发货数_30', 'air_pre实际可发货数_30', 'sea_pre实际可发货数_30', 
       'dhl_pre缺货数', 'air_pre缺货数', 'sea_pre缺货数', 'dhl_pre缺货数_30','air_pre缺货数_30', 'sea_pre缺货数_30',
       # '预占总数', '快递_预占', '空运_预占', '海运_预占', 
       '预占总数', 'FBA自提物流_预占', '快递_预占', '空运_预占', '海运_预占', 
       'CA', 'DE','JP', 'UK', 'US', 
       'TK本地仓', '共享', '本地-在途', '已下单数量','已生产未发货',
       'CA仓搜海外仓', 'DE商易海外仓', 'DE延讯海外仓','JP永翔海外仓', 'UK商易海外仓', 'UK延讯海外仓', 'US商易海外仓', 'US易速达海外仓', '九方欧洲海外仓','元坤海外仓', 'IT永翔海外仓', 'CN易速达:易速达美东GA仓',
       '顺丰SF:美国特拉华S2仓', '顺丰SF:美国洛杉矶S5仓', '顺丰SF:美国达拉斯C1仓', '顺丰SF:美国芝加哥C1仓',
       '7天日均', '15天日均', '30天日均', 
       '断货时间1', '断货总天数1', '损失销量1','断货开始天数1', '断货时间2', '断货总天数2', '损失销量2', '断货开始天数2', '断货时间3', '断货总天数3','损失销量3', '断货开始天数3']


# ret_columns30_color = {'店铺-站点': 'FFD7E3BD', '站点': 'FFD7E3BD', '负责人': 'FFD7E3BD', 'Listing': 'FFD7E3BD', 
#                 '款式': 'FFD7E3BD', 'MSKU': 'FFD7E3BD', '积加SKU': 'FFD7E3BD', 'FNSKU': 'FFD7E3BD', '仓储类型': 'FFD7E3BD','规模定位': 'FFD7E3BD', 
#                 '需求定位': 'FFD7E3BD', '发货定位': 'FFD7E3BD', '备货定位': 'FFD7E3BD', 
#                 '补货': 'FFFDE9D9', '借调': 'FFFDE9D9', '补货_30':'FFFCD5B4', '借调_30': 'FFFCD5B4', 
#                 'FBA在库可售天数': 'FFDAEEF3', 'FBA总可售天数': 'FFDAEEF3', 'FBA在库可售天数_30': 'FFB7DEE8', 'FBA总可售天数_30': 'FFB7DEE8', 
#                 '预估在库日均': 'FFE4DFEC', '预估在库日均_30': 'FFCCC0DA', '在库天数差': 'FFD7E3BD', '在库日均比': 'FFA9D18E','首次断货前可售天数': 'FFF2DCDB', '首次断货前可售天数_30': 'FFE6B8B7',
#                 'FBA库存': 'FFB8CCE4', '已出运': 'FFB8CCE4', 'FBA预占': 'FFB8CCE4','FBA在途': 'FFB8CCE4', '差异量': 'FFE6B8B7',
#                 'dhl_pre': 'FFFDE9D9', 'air_pre': 'FFFDE9D9', 'sea_pre': 'FFFDE9D9', 'dhl_pre_30': 'FFFCD5B4', 'air_pre_30': 'FFFCD5B4', 'sea_pre_30': 'FFFCD5B4', 
#                 'dhl_pre实际可发货数': 'FFDAEEF3', 'air_pre实际可发货数': 'FFDAEEF3','sea_pre实际可发货数': 'FFDAEEF3', 'dhl_pre实际可发货数_30': 'FFB7DEE8', 'air_pre实际可发货数_30': 'FFB7DEE8', 'sea_pre实际可发货数_30': 'FFB7DEE8', 
#                 'dhl_pre缺货数': 'FFE4DFEC', 'air_pre缺货数': 'FFE4DFEC', 'sea_pre缺货数': 'FFE4DFEC', 'dhl_pre缺货数_30': 'FFE4DFEC','air_pre缺货数_30': 'FFE4DFEC', 'sea_pre缺货数_30': 'FFE4DFEC',
#                 'CA': 'FFD7E3BD', 'DE': 'FFD7E3BD','JP': 'FFD7E3BD', 'UK': 'FFD7E3BD', 'US': 'FFD7E3BD', '共享': 'FFD7E3BD', '本地-在途': 'FFF2DCDB', '已下单数量': 'FFE6B8B7',
#                 'CA仓搜海外仓': 'FFB8CCE4', 'DE商易海外仓': 'FFB8CCE4', 'DE延讯海外仓': 'FFB8CCE4','JP永翔海外仓': 'FFB8CCE4', 'UK商易海外仓': 'FFB8CCE4', 'UK延讯海外仓': 'FFB8CCE4', 'US商易海外仓': 'FFB8CCE4', 'US易速达海外仓': 'FFB8CCE4', '九方欧洲海外仓': 'FFB8CCE4','元坤海外仓': 'FFB8CCE4', 
#                 '7天日均': 'FFFCD5B4', '15天日均': 'FFFCD5B4', '30天日均': 'FFFCD5B4', 
#                 '断货时间1': 'FFD7E3BD', '断货总天数1': 'FFD7E3BD', '损失销量1': 'FFD7E3BD','断货开始天数1': 'FFD7E3BD', '断货时间2': 'FFD7E3BD', '断货总天数2': 'FFD7E3BD', '损失销量2': 'FFD7E3BD', '断货开始天数2': 'FFD7E3BD', '断货时间3': 'FFD7E3BD', '断货总天数3': 'FFD7E3BD','损失销量3': 'FFD7E3BD', '断货开始天数3': 'FFD7E3BD'
#                       }

In [15]:
ret_columns60 = ['店铺-站点', '站点', '负责人', 'Listing', '款式', 'MSKU', '积加SKU', 'FNSKU', '仓储类型',
       '规模定位', '需求定位', '发货定位', '备货定位', '总发货天数', '快递在途天数', '空运在途天数', '海运在途天数', '快递打包时间', '空运打包时间', '海运打包时间',
       '补货', '借调', '补货_60','借调_60', 
       'FBA在库可售天数', 'FBA总可售天数', 'FBA在库可售天数_60', 'FBA总可售天数_60', 
       '预估在库日均', '预估在库日均_60', '在库天数差', '在库日均比','首次断货前可售天数', '首次断货前可售天数_60',
       'FBA库存', '已出运', 'FBA预占','FBA在途','差异量',
       'dhl_pre', 'air_pre', 'sea_pre', 'dhl_pre_60', 'air_pre_60', 'sea_pre_60', 
       'dhl_pre实际可发货数', 'air_pre实际可发货数','sea_pre实际可发货数', 'dhl_pre实际可发货数_60', 'air_pre实际可发货数_60', 'sea_pre实际可发货数_60', 
       'dhl_pre缺货数', 'air_pre缺货数', 'sea_pre缺货数', 'dhl_pre缺货数_60','air_pre缺货数_60', 'sea_pre缺货数_60',
       # '预占总数', '快递_预占', '空运_预占', '海运_预占', 
       '预占总数', 'FBA自提物流_预占','快递_预占', '空运_预占', '海运_预占', 
       'CA', 'DE','JP', 'UK', 'US', 
       'TK本地仓', '共享', '本地-在途', '已下单数量','已生产未发货',
       'CA仓搜海外仓', 'DE商易海外仓', 'DE延讯海外仓','JP永翔海外仓', 'UK商易海外仓', 'UK延讯海外仓', 'US商易海外仓', 'US易速达海外仓', '九方欧洲海外仓','元坤海外仓', 'IT永翔海外仓', 'CN易速达:易速达美东GA仓',
       '顺丰SF:美国特拉华S2仓', '顺丰SF:美国洛杉矶S5仓', '顺丰SF:美国达拉斯C1仓', '顺丰SF:美国芝加哥C1仓',
       '7天日均', '15天日均', '30天日均', 
       '断货时间1', '断货总天数1', '损失销量1','断货开始天数1', '断货时间2', '断货总天数2', '损失销量2', '断货开始天数2', '断货时间3', '断货总天数3','损失销量3', '断货开始天数3']

In [16]:
with pd.ExcelWriter(f'../程序建议发货列表/发货列表{parms}-new-7.2.xlsx', engine='openpyxl') as writer:
    ret_df15.to_excel(writer, sheet_name='预估VS15', index=False, columns=ret_columns15)
    ret_df30.to_excel(writer, sheet_name='预估VS30', index=False, columns=ret_columns30)
    ret_df60.to_excel(writer, sheet_name='预估VS60', index=False, columns=ret_columns60)
    shipment_receive_detail.to_excel(writer, sheet_name='接收中-货件详情', index=False)

    # # 获取 Pandas Excel writer 中的 workbook 和 worksheet
    # workbook = writer.book
    # worksheet = writer.sheets['预估VS15']
    # # 使用 Styler 对象为指定列的 header 行添加样式
    # rows = ret_df15.shape[0]
    # for column, color in ret_columns15_color.items():
    #     col_idx = ret_df15.columns.get_loc(column) + 1
    #     # print(col_idx)
    #     for index in range(1, rows+2):
    #         header_cell = worksheet.cell(index, col_idx)
    #         header_cell.fill = openpyxl.styles.PatternFill(start_color=color, end_color=color, fill_type='solid')

    # # 获取 Pandas Excel writer 中的 workbook 和 worksheet
    # workbook = writer.book
    # worksheet = writer.sheets['预估VS30']
    # rows = ret_df30.shape[0]
    # for column, color in ret_columns30_color.items():
    #     col_idx = ret_df30.columns.get_loc(column) + 1
    #     # print(col_idx)
    #     for index in range(1, rows+2):
    #         header_cell = worksheet.cell(index, col_idx)
    #         header_cell.fill = openpyxl.styles.PatternFill(start_color=color, end_color=color, fill_type='solid')

In [17]:
import openpyxl
import warnings